# Task #4: Feature Engineering

**Input:** `data/processed/tao-all2-{train,val,test}.csv` (output of `data_cleaning_preprocessing_pipeline.ipynb`)
**Output:** `data/processed/features-{train,val,test}.csv`, ready for modeling

## Why this step exists
The cleaned data labels each day with that **same day's** risk class. A model trained on that
just reads today's temperature off the row, so it can't warn anyone about anything.
The project goal is **months of lead time**, so this notebook:

1. Rolls daily readings up to **monthly** values per mooring site (ENSO evolves over months; daily data is noisy)
2. Builds a **future target**: the risk class **3 months ahead**
3. Builds **features** using only information available at prediction time (lags, trailing averages, basin-wide signals)
4. Checks a **persistence baseline** ("in 3 months it'll be the same as now") the model must beat

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

LEAD_MONTHS = 3            # how far ahead we forecast
TRAIN_END, VAL_END = 1993, 1996   # same chronological split as the cleaning pipeline

# Run from the repo root whether the kernel starts in notebooks/ (VS Code default) or the root
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: /Users/dharaa/Projects/climate-risk


## 1. Load the three splits and stack them
We stack train/val/test so that lag features at the start of 1994 can look back into late 1993.
We split them apart again at the end, using the same year boundaries.

In [2]:
parts = []
for split in ["train", "val", "test"]:
    d = pd.read_csv(f"data/processed/tao-all2-{split}.csv", parse_dates=["standardized_date"])
    parts.append(d)
daily = pd.concat(parts, ignore_index=True)
print(daily.shape, "|", daily["site_id"].nunique(), "sites |",
      daily["standardized_date"].min().date(), "to", daily["standardized_date"].max().date())
daily.head()

(161073, 30) | 70 sites | 1980-03-07 to 1998-06-23


,obs,year,month,day,latitude,longitude,zonal_wind,meridional_wind,humidity,air_temp,...,temp_mean_30d,temp_std_30d,temp_lag_90,temp_mean_90d,temp_std_90d,zonal_wind_scaled,meridional_wind_scaled,humidity_scaled,air_temp_scaled,ss_temp_scaled
0,1,1980,3,7,-0.02,-109.46,-6.8,0.7,0.0,26.14,...,NaN,NaN,NaN,NaN,NaN,-1.137,0.005,-1.995,-0.249,-0.539
1,2,1980,3,8,-0.02,-109.46,-4.9,1.1,0.0,25.66,...,NaN,NaN,NaN,NaN,NaN,-0.529,0.148,-1.995,-0.512,-0.661
2,3,1980,3,9,-0.02,-109.46,-4.5,2.2,0.0,25.69,...,NaN,NaN,NaN,NaN,NaN,-0.400,0.543,-1.995,-0.495,-0.975
3,4,1980,3,10,-0.02,-109.46,-3.8,1.9,0.0,25.57,...,NaN,NaN,NaN,NaN,NaN,-0.176,0.435,-1.995,-0.561,-1.416
4,5,1980,3,11,-0.02,-109.46,-4.2,1.5,0.0,25.30,...,NaN,NaN,NaN,NaN,NaN,-0.304,0.291,-1.995,-0.709,-1.925


## 2. Recompute the climatology from training years only
The cleaning pipeline's `monthly_clim` averages **all** years, including the 1997–98 test period.
That leaks test information into every anomaly. Here we recompute the "normal" temperature for each
site and calendar month using only **1980–1993**, and derive the anomaly from that.

Humidity sensors didn't exist before 1989, and the pipeline filled those rows with `0`. A real humidity of 0%
is impossible over the ocean, so we turn those back into missing (`NaN`). LightGBM handles missing values natively.

In [3]:
daily["humidity"] = daily["humidity"].where(daily["year"] >= 1989)

def train_climatology(df, col):
    """Average of `col` per (site, calendar month), using training years only."""
    base = df[df["year"] <= TRAIN_END]
    return base.groupby(["site_id", "month"])[col].mean().rename(f"{col}_clim")

for col in ["ss_temp", "zonal_wind", "meridional_wind", "air_temp"]:
    daily = daily.merge(train_climatology(daily, col), on=["site_id", "month"], how="left")

daily["ssta"]            = daily["ss_temp"] - daily["ss_temp_clim"]
daily["zonal_wind_anom"] = daily["zonal_wind"] - daily["zonal_wind_clim"]
daily["mer_wind_anom"]   = daily["meridional_wind"] - daily["meridional_wind_clim"]
daily["air_temp_anom"]   = daily["air_temp"] - daily["air_temp_clim"]

print("Rows with no training-era baseline (site/month never seen before 1994):",
      daily["ssta"].isna().sum())

Rows with no training-era baseline (site/month never seen before 1994): 13362


## 3. Roll up to one row per site per month
We also **fill in missing months** as empty rows. This matters: without it, `shift(1)` on a site with
a 6-month outage would treat "6 months ago" as "last month".

In [4]:
daily["ym"] = daily["standardized_date"].dt.to_period("M")
value_cols = ["ss_temp", "ssta", "zonal_wind", "zonal_wind_anom", "meridional_wind",
              "mer_wind_anom", "air_temp", "air_temp_anom", "humidity"]

monthly = (daily.groupby(["site_id", "ym"])
                .agg(**{c: (c, "mean") for c in value_cols},
                     n_days=("ss_temp", "size"),
                     lat=("nominal_lat", "first"),
                     lon=("nominal_lon", "first"))
                .reset_index())

# Complete calendar per site: every month from that site's first to last observation
def complete_months(g):
    full = pd.period_range(g["ym"].min(), g["ym"].max(), freq="M")
    g = g.set_index("ym").reindex(full)
    g.index.name = "ym"
    g[["lat", "lon"]] = g[["lat", "lon"]].ffill().bfill()
    return g

monthly = (monthly.groupby("site_id", group_keys=True)
                  .apply(complete_months, include_groups=False)
                  .reset_index())
monthly["n_days"] = monthly["n_days"].fillna(0)
monthly["year"]  = monthly["ym"].dt.year
monthly["month"] = monthly["ym"].dt.month
monthly = monthly.sort_values(["site_id", "ym"]).reset_index(drop=True)

print(monthly.shape, "| months with no data (gaps):", (monthly["n_days"] == 0).sum())
monthly.head()

(6571, 16) | months with no data (gaps): 971


,site_id,ym,ss_temp,ssta,zonal_wind,zonal_wind_anom,meridional_wind,mer_wind_anom,air_temp,air_temp_anom,humidity,n_days,lat,lon,year,month
0,-2_-110,1985-06,24.870000,-0.814421,-1.40,3.814737,7.70,4.647368,24.213333,-1.212193,NaN,3.0,-2.0,-110.0,1985,6
1,-2_-110,1985-07,23.230000,-0.568414,-3.79,1.607797,1.05,-1.446916,22.555000,-1.148480,NaN,10.0,-2.0,-110.0,1985,7
2,-2_-110,1985-08,22.686552,-0.043897,-3.37,1.406041,1.57,-0.509714,22.121724,-0.468766,NaN,29.0,-2.0,-110.0,1985,8
3,-2_-110,1985-09,22.030000,-0.576651,-2.79,2.069378,1.66,-0.477512,21.534138,-0.825719,NaN,29.0,-2.0,-110.0,1985,9
4,-2_-110,1985-10,22.073333,-0.785607,-2.85,2.137097,1.42,-1.071244,21.633000,-1.022115,NaN,30.0,-2.0,-110.0,1985,10


## 4. Target: risk class LEAD_MONTHS months ahead
Same 5 NOAA tiers the cleaning pipeline uses, but applied to the anomaly **`LEAD_MONTHS` months in the future**.
`shift(-LEAD_MONTHS)` within each site pulls the future value back onto today's row.

In [5]:
def classify_risk(a):
    if pd.isna(a):   return np.nan
    if a >= 1.5:     return "Extreme Warm"
    if a >= 0.5:     return "Moderate Warm"
    if a <= -1.5:    return "Extreme Cool"
    if a <= -0.5:    return "Moderate Cool"
    return "Neutral"

g = monthly.groupby("site_id")
monthly["target_ssta"] = g["ssta"].shift(-LEAD_MONTHS)          # continuous (for regression)
monthly["target_risk"] = monthly["target_ssta"].map(classify_risk)  # 5-class (headline task)
monthly["current_risk"] = monthly["ssta"].map(classify_risk)       # today's class (used for baseline)

monthly["target_risk"].value_counts(normalize=True).round(3)

target_risk
Neutral          0.553
Moderate Cool    0.184
Moderate Warm    0.158
Extreme Warm     0.056
Extreme Cool     0.049
Name: proportion, dtype: float64

## 5. Features (past and present only)

| Group | Features | Why |
|---|---|---|
| Current state | `ssta`, wind/air-temp anomalies, humidity | Where things stand now |
| Lags | values from 1, 2, 3, 6 months ago | Direction and speed of change |
| Trends | 3- and 6-month trailing means, 1- and 3-month change | Smooths noise, captures momentum |
| Basin-wide | mean anomaly across **all** sites, and across the Niño 3.4 box | El Niño is basin-scale; one buoy sees part of it |
| Season | `sin`/`cos` of month | ENSO events tend to peak Nov–Jan; sin/cos keeps Dec next to Jan |
| Location | `lat`, `lon` | East vs. west Pacific behave differently |

Every group operation is done **per site**, so one buoy's history never leaks into another's.
Rolling windows are trailing (never `center=True`).

In [6]:
g = monthly.groupby("site_id")

lag_cols = ["ssta", "zonal_wind_anom", "mer_wind_anom", "air_temp_anom"]
for col in lag_cols:
    for lag in [1, 2, 3, 6]:
        monthly[f"{col}_lag{lag}"] = g[col].shift(lag)

for col in ["ssta", "zonal_wind_anom"]:
    for w in [3, 6]:
        monthly[f"{col}_roll{w}"] = g[col].transform(lambda s: s.rolling(w, min_periods=2).mean())

monthly["ssta_change1"] = monthly["ssta"] - monthly["ssta_lag1"]
monthly["ssta_change3"] = monthly["ssta"] - monthly["ssta_lag3"]

# Basin-wide signals: one value per month, shared by every site
basin = monthly.groupby("ym").agg(basin_ssta=("ssta", "mean"),
                                  basin_zwind_anom=("zonal_wind_anom", "mean"))
nino34_box = monthly["lat"].between(-5, 5) & monthly["lon"].between(-170, -120)
basin["nino34_ssta"] = monthly[nino34_box].groupby("ym")["ssta"].mean()
basin["nino34_ssta_roll3"] = basin["nino34_ssta"].rolling(3, min_periods=2).mean()
monthly = monthly.merge(basin, on="ym", how="left")

monthly["month_sin"] = np.sin(2 * np.pi * monthly["month"] / 12)
monthly["month_cos"] = np.cos(2 * np.pi * monthly["month"] / 12)

FEATURES = (["ssta", "zonal_wind_anom", "mer_wind_anom", "air_temp_anom", "humidity", "lat", "lon",
             "month_sin", "month_cos", "ssta_change1", "ssta_change3"]
            + [c for c in monthly.columns if "_lag" in c or "_roll" in c]
            + ["basin_ssta", "basin_zwind_anom", "nino34_ssta", "nino34_ssta_roll3"])
FEATURES = list(dict.fromkeys(FEATURES))
print(len(FEATURES), "features")

35 features


## 6. Split by year, with a gap at each boundary
A row from Dec 1993 (train) has a target from Mar 1994, which is inside validation. To keep
validation truly unseen, we drop the last `LEAD_MONTHS` months of train, and likewise of validation.
We also drop rows with no target (the end of each site's record, and data gaps).

In [7]:
t = monthly["ym"].dt.to_timestamp()
labeled = monthly["target_risk"].notna() & monthly["ssta"].notna()

def split_mask(start_year, end_year):
    in_range = monthly["year"].between(start_year, end_year)
    target_in_range = (t + pd.DateOffset(months=LEAD_MONTHS)).dt.year <= end_year
    return in_range & target_in_range & labeled

train = monthly[split_mask(1980, TRAIN_END)]
val   = monthly[split_mask(TRAIN_END + 1, VAL_END)]
test  = monthly[split_mask(VAL_END + 1, 2100)]
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} {len(d):5d} rows  {d['ym'].min()} to {d['ym'].max()}")

train  1988 rows  1980-08 to 1993-09
val    1529 rows  1994-01 to 1996-09
test    694 rows  1997-01 to 1998-03


## 7. Sanity checks

**Which features move with the future anomaly?** Correlation with `target_ssta` on the training set only.

In [8]:
corr = train[FEATURES + ["target_ssta"]].corr(numeric_only=True)["target_ssta"].drop("target_ssta")
corr.sort_values(key=abs, ascending=False).round(3).head(15)

ssta                  0.648
basin_ssta            0.597
ssta_roll3            0.586
nino34_ssta           0.562
ssta_lag1             0.548
air_temp_anom         0.541
nino34_ssta_roll3     0.508
ssta_roll6            0.475
air_temp_anom_lag1    0.466
ssta_lag2             0.448
air_temp_anom_lag2    0.389
ssta_lag3             0.347
basin_zwind_anom      0.333
ssta_change3          0.326
air_temp_anom_lag3    0.317
Name: target_ssta, dtype: float64

**Persistence baseline:** predict that the risk class in 3 months equals today's.
This is the number the model has to beat (on validation, Macro F1).

In [9]:
base = val.dropna(subset=["current_risk"])
print("Persistence baseline  Macro F1 (val):",
      round(f1_score(base["target_risk"], base["current_risk"], average="macro"), 3))
print("Majority-class baseline Macro F1 (val):",
      round(f1_score(base["target_risk"], ["Neutral"] * len(base), average="macro"), 3))
print()
print(classification_report(base["target_risk"], base["current_risk"], zero_division=0))

Persistence baseline  Macro F1 (val): 0.353
Majority-class baseline Macro F1 (val): 0.132

               precision    recall  f1-score   support

 Extreme Cool       0.29      0.26      0.28       110
 Extreme Warm       0.00      0.00      0.00        17
Moderate Cool       0.47      0.45      0.46       408
Moderate Warm       0.38      0.39      0.38       237
      Neutral       0.64      0.66      0.65       757

     accuracy                           0.52      1529
    macro avg       0.36      0.35      0.35      1529
 weighted avg       0.52      0.52      0.52      1529



## 8. Save
Each file keeps identifiers (`site_id`, `ym`), the features, and both targets.

In [10]:
keep = ["site_id", "ym", "year", "month"] + FEATURES + ["current_risk", "target_ssta", "target_risk"]
for name, d in [("train", train), ("val", val), ("test", test)]:
    out = f"data/processed/features-{name}.csv"
    d[keep].to_csv(out, index=False)
    print("saved", out, d[keep].shape)

saved data/processed/features-train.csv (1988, 42)
saved data/processed/features-val.csv (1529, 42)
saved data/processed/features-test.csv (694, 42)
